# 03 — Model Experiments

Unified experiment workflow for all four candidate models:
- **Ridge Regression** — Linear baseline
- **Random Forest** — Ensemble tree
- **XGBoost** — Gradient boosting
- **LSTM** — Recurrent neural network

All models import from `src/models/`. No model code is duplicated in this notebook.

**Evaluation metrics:** MAE, RMSE, R² per horizon (24h, 48h, 72h) + overall  
**Selection criteria:** Performance vs complexity tradeoff

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Import LSTM from production code — no duplication
from src.models.lstm_model import LSTMModel

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

TARGET_COLS = ['target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']
EXCLUDE = ['timestamp', 'location_id', 'city_name', 'data_source', 'aqi_category',
           'aqi_standard', 'aqi_method', 'aqi_method_version', 'aqi_source']
RANDOM_SEED = 42

## 1. Load and Prepare Data

In [ ]:
train_feat = pd.read_csv('../data/processed/train_features.csv')
train_tgt = pd.read_csv('../data/processed/train_targets.csv')
val_feat = pd.read_csv('../data/processed/val_features.csv')
val_tgt = pd.read_csv('../data/processed/val_targets.csv')
test_feat = pd.read_csv('../data/processed/test_features.csv')
test_tgt = pd.read_csv('../data/processed/test_targets.csv')

feature_cols = [c for c in train_feat.columns if c not in EXCLUDE
                and train_feat[c].dtype in ['float64', 'int64', 'bool']]

def prepare(feat, tgt):
    mask = tgt[TARGET_COLS].notna().all(axis=1)
    X = feat.loc[mask, feature_cols].fillna(0).values
    y = tgt.loc[mask, TARGET_COLS].values
    return X, y

X_train, y_train = prepare(train_feat, train_tgt)
X_val, y_val = prepare(val_feat, val_tgt)
X_test, y_test = prepare(test_feat, test_tgt)

print(f"Features: {len(feature_cols)}")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

## 2. Unified Evaluation Helper

In [ ]:
def evaluate(y_true, y_pred, model_name):
    """Evaluate multi-output predictions per horizon."""
    results = []
    for i, h in enumerate([24, 48, 72]):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        results.append({'Model': model_name, 'Horizon': f'{h}h',
                        'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)})
    mae_all = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    rmse_all = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
    r2_all = r2_score(y_true.flatten(), y_pred.flatten())
    results.append({'Model': model_name, 'Horizon': 'All',
                    'MAE': round(mae_all, 2), 'RMSE': round(rmse_all, 2), 'R²': round(r2_all, 4)})
    return pd.DataFrame(results)

def train_and_evaluate(model_name, model, X_tr, y_tr, X_te, y_te, use_scaled=False):
    """Train model, predict, evaluate, and return results row."""
    Xtr = X_tr_s if use_scaled else X_tr
    Xte = X_te_s if use_scaled else X_te

    t0 = time.time()
    model.fit(Xtr, y_tr)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(Xte)
    infer_time = (time.time() - t0) / len(X_te) * 1000

    metrics = evaluate(y_te, y_pred, model_name)
    metrics['Train Time (s)'] = round(train_time, 3)
    metrics['Infer Latency (ms)'] = round(infer_time, 4)
    return metrics, model, y_pred

## 3. Ridge Regression (Baseline)

In [ ]:
ridge_model = Ridge(alpha=1.0, random_state=RANDOM_SEED)
ridge_results, ridge_fitted, y_pred_ridge = train_and_evaluate(
    'Ridge', ridge_model, X_train, y_train, X_test, y_test, use_scaled=True
)
print(ridge_results.to_string(index=False))

## 4. Random Forest

In [ ]:
rf_model = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=100, max_depth=20,
                          random_state=RANDOM_SEED, n_jobs=-1)
)
rf_results, rf_fitted, y_pred_rf = train_and_evaluate(
    'RandomForest', rf_model, X_train, y_train, X_test, y_test
)
print(rf_results.to_string(index=False))

## 5. XGBoost

In [ ]:
xgb_model = MultiOutputRegressor(
    xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                     random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)
)
xgb_results, xgb_fitted, y_pred_xgb = train_and_evaluate(
    'XGBoost', xgb_model, X_train, y_train, X_test, y_test
)
print(xgb_results.to_string(index=False))

## 6. LSTM

Uses `LSTMModel` from `src/models/lstm_model.py` — no code duplication.  
LSTM requires sequential input (24-timestep windows).

In [ ]:
SEQ_LEN = 24
lstm_model = LSTMModel(
    sequence_length=SEQ_LEN,
    n_features=X_train.shape[1],
    n_targets=y_train.shape[1],
    lstm_units=[64, 32],
    dropout_rate=0.2,
    learning_rate=0.001,
    random_seed=RANDOM_SEED,
)

t0 = time.time()
lstm_history = lstm_model.fit(
    X_train, y_train, X_test, y_test,
    epochs=30, batch_size=64, verbose=0,
)
lstm_train_time = time.time() - t0

t0 = time.time()
y_pred_lstm_raw = lstm_model.predict(X_test)
lstm_infer_time = (time.time() - t0) / max(len(y_pred_lstm_raw), 1) * 1000

# Align: LSTM predictions start at seq_len-1
y_test_aligned = y_test[SEQ_LEN - 1:SEQ_LEN - 1 + len(y_pred_lstm_raw)]

lstm_results = evaluate(y_test_aligned, y_pred_lstm_raw, 'LSTM')
lstm_results['Train Time (s)'] = round(lstm_train_time, 1)
lstm_results['Infer Latency (ms)'] = round(lstm_infer_time, 4)
print(f"Epochs trained: {lstm_history.get('epochs_trained', 0)}")
print(lstm_results.to_string(index=False))

## 7. Unified Comparison Table

In [ ]:
all_results = pd.concat([ridge_results, rf_results, xgb_results, lstm_results], ignore_index=True)

# Pivot for clean comparison
comparison = all_results[all_results['Horizon'] != 'All'].copy()
comparison = comparison.pivot_table(
    index='Model', columns='Horizon',
    values=['MAE', 'RMSE', 'R²'],
    aggfunc='first'
)
comparison.columns = [f'{v}_{h}' for v, h in comparison.columns]

# Add overall and timing
overall = all_results[all_results['Horizon'] == 'All'].set_index('Model')
comparison['Overall_MAE'] = overall['MAE']
comparison['Overall_RMSE'] = overall['RMSE']
comparison['Overall_R2'] = overall['R²']
comparison['Train_Time_s'] = overall['Train Time (s)']
comparison['Infer_ms'] = overall['Infer Latency (ms)']

print("=" * 80)
print("UNIFIED MODEL COMPARISON")
print("=" * 80)
comparison.round(4)

In [ ]:
# Clean summary table
summary_rows = []
for model_name in ['Ridge', 'RandomForest', 'XGBoost', 'LSTM']:
    m = all_results[(all_results['Model'] == model_name) & (all_results['Horizon'] == 'All')]
    if not m.empty:
        row = m.iloc[0]
        summary_rows.append({
            'Model': model_name,
            'MAE': row['MAE'],
            'RMSE': row['RMSE'],
            'R²': row['R²'],
            'Train Time (s)': row['Train Time (s)'],
            'Infer Latency (ms)': row['Infer Latency (ms)'],
        })
summary = pd.DataFrame(summary_rows)
summary

## 8. Per-Horizon MAE Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
horizons = ['24h', '48h', '72h']
models = ['Ridge', 'RandomForest', 'XGBoost', 'LSTM']
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

for ax, h in zip(axes, horizons):
    h_data = all_results[(all_results['Horizon'] == h) & (all_results['Model'].isin(models))]
    bars = ax.bar(h_data['Model'], h_data['MAE'], color=colors[:len(h_data)], edgecolor='white')
    ax.set_title(f'MAE — {h}')
    ax.set_ylabel('MAE')
    for bar, val in zip(bars, h_data['MAE']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison: MAE per Horizon', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. R² Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, h in zip(axes, horizons):
    h_data = all_results[(all_results['Horizon'] == h) & (all_results['Model'].isin(models))]
    bars = ax.bar(h_data['Model'], h_data['R²'], color=colors[:len(h_data)], edgecolor='white')
    ax.set_title(f'R² — {h}')
    ax.set_ylabel('R²')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, h_data['R²']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison: R² per Horizon', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 10. Training Time & Inference Latency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

time_data = summary.set_index('Model')
time_data[['Train Time (s)']].plot(kind='bar', ax=axes[0], color=colors[:4], legend=False, edgecolor='white')
axes[0].set_title('Training Time')
axes[0].set_ylabel('Seconds')
axes[0].tick_params(axis='x', rotation=0)

time_data[['Infer Latency (ms)']].plot(kind='bar', ax=axes[1], color=colors[:4], legend=False, edgecolor='white')
axes[1].set_title('Inference Latency')
axes[1].set_ylabel('ms per sample')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 11. Complexity vs Performance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, row in summary.iterrows():
    ax.scatter(row['Train Time (s)'], row['MAE'], s=200, c=colors[i],
              label=row['Model'], zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(row['Model'], (row['Train Time (s)'], row['MAE']),
               textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Training Time (s)')
ax.set_ylabel('Overall MAE')
ax.set_title('Model Complexity vs Performance')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()